# 3. Tropicalization and toric degenerations

This notebook follows Section 4 of the extended abstract. It covers

1. how the tropical $E_6$-Grassmannian is computed with `gfan`, starting from an $(E_6,P_6)$-matroid,
2. the resulting fan and the statements of Theorem 4.1,
3. matroid subdivisions of $2_{21}$,
4. prime cones and toric degenerations (Theorem 4.2), including the Hibi and the chain-polytope degenerations.

Two computations are too long for a notebook and are done by scripts:

| computation | script | result |
|---|---|---|
| the tropical variety (gfan, about five minutes) | `scripts/gfan_input.jl`, `scripts/gfan_traverse.sh` | `data/gfan/cayley_plane_trop.dat` |
| matroid subdivisions for all $14\,155$ orbits of cones | `scripts/matroid_subdivisions.jl` | printed summary |
| the $60$ toric degenerations and their polytopes | `scripts/toric_degenerations.jl` | `data/toric_degenerations.csv` |

The functions are defined in `src/tropical.jl` and `src/toric.jl`. We use the max convention: $\operatorname{in}_w f$ collects the terms of $f$ of maximal $w$-weight.

In [1]:
include("../src/E6P6CoxeterMatroids.jl");

  ___   ___   ___    _    ____
 / _ \ / __\ / __\  / \  |  _ \  | Combining and extending ANTIC, GAP,
| |_| |\__ \| |__  / ^ \ |  ´ /  | Polymake and Singular
 \___/ \___/ \___//_/ \_\|_|\_\  | Type "?Oscar" for more information
o--------o-----o-----o--------o  | Documentation: https://docs.oscar-system.org
  S Y M B O L I C   T O O L S    | Version 1.8.2


## Computing the tropical variety with gfan

Let $I=\langle D_y\rangle$ be the ideal of $E_6/P_6$. The traversal algorithm of gfan needs a maximal cone of $\operatorname{Trop}(X^\circ)$ to start from, that is, a weight $w$ in the tropical variety whose Gröbner cone has the maximal dimension $17$ in $\mathbb{R}^{27}$. A random search in $27$ variables is hopeless.

Here the classification helps. By Theorem 2.4(iii), the indicator vector of a matroid never truncates a quadric $D_y$ to a single monomial. So we look among the $230$ orbit representatives for matroids whose indicator vector induces a fine enough subdivision of $2_{21}$: the secondary cone of the subdivision should have dimension at least $17$. One of the candidates is `Q_start` (orbit number $111$), and gfan confirms in Step 2 below that it works.

In [ ]:
candidates = [Q for Q in E6_mats if
    dim(secondary_cone(subdivision_of_points(Gosset_lines, -matroid_index_to_weight_vector(Q)))) >= 17]
@show length(candidates)
@show Q_start in candidates
println(line_labels[Q_start])

The initial ideal for this weight contains no monomial, so the weight lies in the tropical variety.

In [ ]:
w = matroid_index_to_weight_vector(Q_start)
J = in_w(w)
@show !isone(saturate_all(J, abc));

gfan is run outside Julia, in three steps.

**Step 1.** `scripts/gfan_input.jl` writes the ideal and the starting weight to `data/gfan/cayley_plane_init.dat`, and the generators of $W(E_6)$ to `data/gfan/symmetries.dat`. The variables are renamed $x_{01},\dots,x_{27}$. The group acts on the coordinates by *signed* permutations; the six simple reflections are stored in `signed_symmetries`, and one can check that they preserve $I$.

In [ ]:
print(gfan_ideal_input(w)[1:330], " ...")

In [ ]:
@show all(hom(R, R, s)(I) == I for s in signed_symmetries)
print(gfan_symmetry_input())

**Step 2.** `gfan_initialforms --ideal --pair` computes the starting cone: the initial ideal $\operatorname{in}_w I$ together with a compatible reduced Gröbner basis of $I$.

**Step 3.** `gfan_tropicaltraverse --symmetry --symsigns --nocones` traverses the tropical variety up to symmetry.

```sh
cd data/gfan
gfan_initialforms --ideal --pair < cayley_plane_init.dat > cayley_plane_cone.dat
cat symmetries.dat >> cayley_plane_cone.dat
gfan_tropicaltraverse --symmetry --symsigns --nocones < cayley_plane_cone.dat > cayley_plane_trop.dat
```

These are the commands of `scripts/gfan_traverse.sh`. With gfan 0.6.2 the traversal takes about five minutes.

## The tropical $E_6$-Grassmannian (Theorem 4.1)

The output file is read when the code is loaded:

* `trop_OP2`: the sections of the file,
* `rays_OP2`: the $12\,885$ rays, modulo the lineality space,
* `cones_OP2`: one representative of each $W(E_6)$-orbit of cones, as a list of ray indices (the first one is the empty cone),
* `max_cones_OP2`: the $421$ orbits of maximal cones.

gfan works in $\mathbb{R}^{27}$, where the fan has dimension $17$ and lineality dimension $7$; modulo $\mathbb{R}\mathbf{1}$ these are the numbers $16$ and $6$ of the abstract.

In [ ]:
for key in ["AMBIENT_DIM", "DIM", "LINEALITY_DIM", "N_RAYS", "SIMPLICIAL", "PURE", "F_VECTOR"]
    println(rpad(key, 14), trop_OP2[key][1])
end
@show unique(trop_OP2["MULTIPLICITIES_ORBITS"]);

In [ ]:
# the f-vector up to W(E6)
@show [count(c -> length(c) == k, cones_OP2) for k in 1:10]
@show length(cones_OP2) - 1;

The lineality space consists of the affine-linear functions on $2_{21}$. We compute it from a Gröbner basis of $I$ and compare it with the span of the coordinate functions of `Gosset_lines`.

In [ ]:
L = lineality_space_V_rep(I)
@show rank(L)
@show rank(vcat(L, transpose(matrix(QQ, Gosset_lines))));

The ten orbits of rays. The two orbits of size $36$ are spanned, modulo lineality, by the indicator vectors of the $36$ double sixes and of their complements.

In [ ]:
orbits = ray_orbits()
@show length.(orbits)

double_six = matroid_index_to_weight_vector(1:12)          # a1,...,a6,b1,...,b6
for o in orbits
    length(o) == 36 || continue
    println("contains the double six: ", any(same_ray(rays_OP2[i], double_six) for i in o),
            ",  its complement: ", any(same_ray(rays_OP2[i], -double_six) for i in o))
end

## Matroid subdivisions

Every $w\in\operatorname{Trop}(X^\circ)$ induces a regular subdivision of $2_{21}$. `induced_subdivision(w)` returns its maximal cells, and `vec_rep(c)` is a weight in the relative interior of the cone `c`. All cells are $(E_6,P_6)$-matroid polytopes.

In [ ]:
c = max_cones_OP2[1]
cells = induced_subdivision(vec_rep(c))
@show length(cells)
@show all(is_E6P6_matroid, cells)
println(line_labels[cells[1]])

For the $421$ orbits of maximal cones (a few minutes). The script `scripts/matroid_subdivisions.jl` does the same for all $14\,155$ orbits of cones.

In [ ]:
subdivisions = [induced_subdivision(vec_rep(c)) for c in max_cones_OP2]
@show all(all(is_E6P6_matroid, cells) for cells in subdivisions)
ncells = length.(subdivisions)
[n => count(==(n), ncells) for n in sort(unique(ncells))]

## Toric degenerations (Theorem 4.2)

A maximal cone is *prime* if $\operatorname{in}_w I$ is a prime ideal for $w$ in its relative interior. Then $\operatorname{in}_w I$ is a toric ideal, and $E_6/P_6$ degenerates flatly to a projective toric variety.

We start with orbit number $88$. Its initial ideal is a binomial ideal.

In [ ]:
J = in_w(vec_rep(max_cones_OP2[88]))
gens(J)

A binomial ideal containing no monomial is prime over $\mathbb{C}$ if and only if its lattice of exponent differences is saturated and it is saturated with respect to the variables (Eisenbud–Sturmfels). This is what `is_prime_binomial` tests; it agrees with Oscar's general primality test.

In [ ]:
@show is_prime_binomial(J)
@show is_prime(J);

The toric variety $V(\operatorname{in}_w I)$ is given by $27$ lattice points, the columns of the matrix `A`: the rows of `A` form a basis of the integer vectors orthogonal to all exponent differences (normalized so that the first point is the origin). Their convex hull is the polytope $\Delta_\sigma\subseteq\mathbb{R}^{16}$.

In [ ]:
A, Dmat = toric_configuration_matrix(J)
@show size(A), rank(Dmat)
P = convex_hull(matrix(QQ, toric_points(A)))
@show dim(P), n_vertices(P);

The invariants of Theorem 4.2 are computed with Oscar's polyhedral geometry: the only lattice points are the $27$ vertices, the normalized volume is $78=\deg E_6/P_6$, the polytope is normal, and its $h^*$-vector $(1,10,28,28,10,1)$ is palindromic. Hence $\Delta_\sigma$ is Gorenstein of codegree $17-5=12$, and the toric variety is a projectively normal Gorenstein Fano variety with $\omega=\mathcal{O}(-12)$.

In [ ]:
t = toric_invariants(P)

### The Hibi degeneration and the chain polytope

Order the $27$ weights by $\lambda\le\mu$ when $\mu-\lambda$ is a nonnegative combination of simple roots (the matrix `leq`). This is a distributive lattice with $16$ join-irreducibles. Its order ideals give the vertices of the order polytope, `order_polytope_points`, and its antichains those of the chain polytope, `chain_polytope_points`.

`lattice_equivalence` searches for a bijection between two point configurations that preserves all affine dependencies over $\mathbb{Z}$. Orbit $88$ is the Hibi degeneration.

In [ ]:
@show length(join_irreducibles)
s = lattice_equivalence(toric_points(A), order_polytope_points)

Orbits $313$ and $314$ give the chain polytope.

In [ ]:
for i in [313, 314]
    Ai, _ = toric_configuration_matrix(in_w(vec_rep(max_cones_OP2[i])))
    println(i, ":  chain polytope ", !isnothing(lattice_equivalence(toric_points(Ai), chain_polytope_points)),
               ",  order polytope ", !isnothing(lattice_equivalence(toric_points(Ai), order_polytope_points)))
end

### All prime cones

Testing all $421$ orbits takes a while; this is done in `scripts/toric_degenerations.jl`, which writes one row per prime orbit to `data/toric_degenerations.csv`. To run the test here, uncomment the next cell.

In [ ]:
# prime = [i for i in 1:421 if is_prime_binomial(in_w(vec_rep(max_cones_OP2[i])))]
# length(prime)

In [ ]:
rows = [split(l, ",") for l in readlines(joinpath(datadir, "toric_degenerations.csv"))[2:end]]
@show length(rows)
@show unique(r[3] for r in rows), unique(r[4] for r in rows), unique(r[5] for r in rows), unique(r[6] for r in rows)
@show count(r -> r[7] == "true", rows)                      # generated by quadrics
@show [parse(Int, r[1]) for r in rows if r[8] == "order"], [parse(Int, r[1]) for r in rows if r[8] == "chain"];

Table 2: the $60$ toric degenerations by the number of facets of their polytopes.

In [ ]:
facets = [parse(Int, r[2]) for r in rows]
[n => count(==(n), facets) for n in sort(unique(facets))]